# Kopfdetektor (YOLO11s, CrowdHuman) auf dem Problem-Ausschnitt

**Frage:** Findet ein **Kopfdetektor** den Kopf des Patienten dort, wo die Gesichtsdetektoren SCRFD-10G und SCRFD-34G nichts finden (Frames 8100–8249 von VID_76)?

**Unterschied zu SCRFD:** SCRFD sucht ein *Gesicht*. Ein Kopfdetektor sucht den *ganzen Kopf*, auch im Profil, von hinten oder im Kissen. Die Boxen sind deshalb größer, und es wird mehr verpixelt.

**Was gleich bleibt:** Die Erkennung liefert wieder Boxen `[x, y, Breite, Höhe, Score]` pro Frame. Tracking, Lücken füllen, Glätten und Verpixeln sind deshalb unverändert. Nur das Modell ist ein anderes.

**Datenschutz:** Das Notebook gibt nur Zahlen aus. Videos landen in `data/processed` (gitignored). Vor einem Commit alle Ausgaben löschen.

## 0. Das Modell und worauf zu achten ist

| | |
|---|---|
| Modell | `yolov11_phd_s.onnx`, YOLOv11-small, Klassen **0 = Person, 1 = Kopf** |
| Quelle | Hugging Face `Sharath33/Person` (Drittanbieter, trainiert auf **CrowdHuman**) |
| SHA-256 | `3f0a061b37f1398be76fc344c116a8a6c42fb835583cabe33ddd80812ef4af10` |
| Lizenz | laut Seite *openrail++* (Volltext nicht geprüft). CrowdHuman ist ein Forschungsdatensatz. Vor einer Weitergabe klären. |
| Eingabe | fest **640 × 640** (anders als SCRFD kein beliebiges Format) |
| Ausgabe | `boxes` (8400 × 4, Mitte x, Mitte y, Breite, Höhe in 640er-Pixeln), `scores`, `classes`. **Ohne NMS**, wir machen das selbst. |

**Drei Abweichungen vom Beispielskript des Autors** (`models/head_phd/inference_reference.py`, nur gelesen, nie ausgeführt):
1. Sein Skript erwartet ein anderes Ausgabeformat (`[1, 6, 8400]`) als die Datei liefert. Wir schreiben die Auswertung selbst.
2. Sein Skript sagt „BGR“. Tatsächlich liefert **RGB** deutlich höhere Scores. Das prüft Abschnitt 6 an einem Frame nach.
3. Letterbox mit grauem Rand 114 (YOLO-Standard), nicht schwarz wie bei SCRFD.

**Wichtig:** Die Scores eines YOLO-Modells sind nicht mit SCRFD-Scores vergleichbar. Ein Score von 0,5 heißt hier etwas anderes. Deshalb testen wir mehrere Startschwellen.

## 1. Setup und Einstellungen

In [ ]:
import pickle
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

PROJECT_ROOT = Path.cwd().parent
MODEL_HEAD = PROJECT_ROOT / "models" / "head_phd" / "yolov11_phd_s.onnx"
MODEL_10G = PROJECT_ROOT / "models" / "buffalo_l" / "buffalo_l" / "det_10g.onnx"      # nur für die Kontrolle der Frame-Nummer
INPUT_VIDEO = PROJECT_ROOT / "data" / "raw" / "PAT_1" / "VID_76.AVI"
FACES_10G_FILE = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "VID_76_faces_1280.pkl"   # fertige 10G-Erkennung (ganzes Video)

# Derselbe Ausschnitt wie beim 34G-Test: Frames CLIP_START bis CLIP_END-1 (150 Frames = 6 Sekunden).
CLIP_START, CLIP_END = 8100, 8250

HEADS_FILE = PROJECT_ROOT / "data" / "interim" / "PAT_1" / f"VID_76_heads_yolo11s_{CLIP_START}_{CLIP_END}.pkl"
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "PAT_1"

FPS = 25
SPLIT = 960      # Grenze zwischen linker (RGB) und rechter (schwarz-weiß) Ansicht in Pixeln

for name, p in [("Kopfmodell", MODEL_HEAD), ("Modell 10G", MODEL_10G), ("Video", INPUT_VIDEO), ("10G-Erkennungen", FACES_10G_FILE)]:
    print(f"{name}: {'gefunden' if p.exists() else 'FEHLT'}")

## 2. Das Modell ansehen
Erwartet: Eingang `[batch, 3, 640, 640]`, drei Ausgänge `boxes`, `scores`, `classes` mit je 8400 Einträgen (so viele Gitterpunkte hat ein 640 × 640-Bild bei den Schritten 8, 16 und 32: 80·80 + 40·40 + 20·20 = 8400).

In [ ]:
s = ort.InferenceSession(str(MODEL_HEAD), providers=["CPUExecutionProvider"])
print(MODEL_HEAD.name, f"{MODEL_HEAD.stat().st_size / 1e6:.1f} MB")
for i in s.get_inputs():
    print("  Eingang:", i.name, i.shape)
for o in s.get_outputs():
    print("  Ausgang:", o.name, o.shape)
print("Gitterpunkte:", 80 * 80 + 40 * 40 + 20 * 20)

## 3. Der Kopfdetektor

Schritte pro Bild:
1. **Letterbox:** auf 640 × 640 verkleinern, Seitenverhältnis behalten, den Rest mit Grau (114) auffüllen. Dein Video (1920 × 1080) wird auf 640 × 360 verkleinert, also auf **ein Drittel**.
2. **Farben:** OpenCV liefert BGR, das Modell erwartet RGB. Dann durch 255 teilen.
3. **Netz laufen lassen:** 8400 Kandidaten mit Box, Score und Klasse.
4. **Filtern:** nur Klasse „Kopf“ (1) und Score ≥ Schwelle.
5. **Zurückrechnen:** von der Mitte-Form `cx, cy, w, h` in 640er-Pixeln auf `x, y, w, h` im Originalbild (Rand abziehen, durch den Verkleinerungsfaktor teilen).
6. **NMS:** überlappende Boxen desselben Kopfes zu einer zusammenfassen (IoU 0,6, Wert vom Autor).

Das Ergebnis hat dasselbe Format wie bei SCRFD, damit das Tracking unverändert funktioniert.

In [ ]:
class HeadDetector:
    def __init__(self, model_path, score_threshold=0.2, nms_threshold=0.6, class_id=1, rgb=True,
                 size=640, pad_value=114):
        self.score_threshold = score_threshold
        self.nms_threshold = nms_threshold
        self.class_id = class_id          # 0 = Person, 1 = Kopf
        self.rgb = rgb                    # True = RGB (richtig), False = BGR (nur zum Vergleich)
        self.size = size
        self.pad_value = pad_value
        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name

    def detect(self, img_bgr):
        ih, iw = img_bgr.shape[:2]

        # 1. Letterbox auf size x size
        scale = min(self.size / iw, self.size / ih)
        new_w, new_h = int(iw * scale), int(ih * scale)
        top, left = (self.size - new_h) // 2, (self.size - new_w) // 2
        canvas = np.full((self.size, self.size, 3), self.pad_value, dtype=np.uint8)
        canvas[top:top + new_h, left:left + new_w] = cv2.resize(img_bgr, (new_w, new_h))

        # 2. Farben und Wertebereich
        if self.rgb:
            canvas = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)
        blob = np.ascontiguousarray(canvas.transpose(2, 0, 1)[None]).astype(np.float32) / 255.0

        # 3. Netz
        boxes, scores, classes = self.session.run(None, {self.input_name: blob})
        boxes = boxes[0]                                   # (8400, 4)
        scores = scores[0].reshape(-1)                     # (8400,)
        classes = np.rint(classes[0].reshape(-1)).astype(int)

        # 4. Filtern
        keep = (classes == self.class_id) & (scores >= self.score_threshold)
        if not keep.any():
            return 0, np.empty((0, 5), dtype=np.float32)
        b, sc = boxes[keep], scores[keep]

        # 5. Zurückrechnen: Mitte -> linke obere Ecke, Rand abziehen, durch Faktor teilen
        x = (b[:, 0] - b[:, 2] / 2 - left) / scale
        y = (b[:, 1] - b[:, 3] / 2 - top) / scale
        w, h = b[:, 2] / scale, b[:, 3] / scale
        xywh = np.stack([x, y, w, h], axis=1)

        # 6. NMS
        idxs = np.array(cv2.dnn.NMSBoxes(xywh.tolist(), sc.tolist(), self.score_threshold, self.nms_threshold)).reshape(-1)
        faces = np.array([[*xywh[i], sc[i]] for i in idxs], dtype=np.float32)
        return len(faces), faces

## 4. Hilfsfunktionen: Tracking, Lücken, Glätten (unverändert aus dem 10G-Notebook)

Hier nichts ändern, damit der Vergleich fair bleibt.

In [ ]:
def iou(a, b):
    """Überlappung zweier Boxen (x, y, w, h): 0 = keine, 1 = identisch."""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0


def inter_over_smaller(a, b):
    """Anteil der kleineren Box, der in der anderen liegt (0 bis 1)."""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / max(1e-9, min(aw * ah, bw * bh))


def near_track(box, tracks, pad=0.5, min_frac=0.3):
    """True, wenn mindestens min_frac der Box im verpixelten Bereich einer Spur liegt."""
    x, y, w, h = box
    for t in tracks:
        tx, ty, tw, th = t["box"]
        rx0, ry0 = tx - pad * tw, ty - pad * th
        rx1, ry1 = tx + tw + pad * tw, ty + th + pad * th
        ix = max(0, min(x + w, rx1) - max(x, rx0))
        iy = max(0, min(y + h, ry1) - max(y, ry0))
        if ix * iy >= min_frac * w * h:
            return True
    return False


def track_with_ids(all_faces, start_threshold=0.5, low_threshold=0.2, match_iou=0.3,
                   max_age=8, damping=0.85, near_threshold=None):
    tracks, result, next_id = [], [], 0
    for faces in all_faces:
        cands = [(tuple(float(v) for v in f[:4]), float(f[4])) for f in faces if f[4] >= low_threshold]

        for t in tracks:                                    # erwartete Position
            x, y, w, h = t["box"]
            t["pred"] = (x + t["v"][0], y + t["v"][1], w, h)

        pairs = sorted(((iou(t["pred"], box), ti, ci)
                        for ti, t in enumerate(tracks) for ci, (box, s) in enumerate(cands)),
                       reverse=True)
        used_t, used_c = set(), set()
        for ov, ti, ci in pairs:
            if ov < match_iou:
                break
            if ti in used_t or ci in used_c:
                continue
            t, new = tracks[ti], cands[ci][0]
            steps = t["age"] + 1
            mvx = (new[0] - t["det"][0]) / steps
            mvy = (new[1] - t["det"][1]) / steps
            t["v"] = (0.5 * t["v"][0] + 0.5 * mvx, 0.5 * t["v"][1] + 0.5 * mvy)
            t["box"], t["det"], t["age"] = new, new, 0
            used_t.add(ti)
            used_c.add(ci)

        for ti, t in enumerate(tracks):                     # nicht gefunden: weiterlaufen
            if ti not in used_t:
                t["box"] = t["pred"]
                t["v"] = (t["v"][0] * damping, t["v"][1] * damping)
                t["age"] += 1
        tracks = [t for t in tracks if t["age"] <= max_age]

        for ci, (box, s) in enumerate(cands):               # neue Spuren
            if ci in used_c:
                continue
            if s < start_threshold:
                # NEU: schwacher Fund nur, wenn er an einer bestehenden Spur liegt
                if near_threshold is None or s < near_threshold or not near_track(box, tracks):
                    continue
            if any(inter_over_smaller(box, t["box"]) >= 0.5 for t in tracks):
                continue
            tracks.append({"id": next_id, "box": box, "det": box, "v": (0.0, 0.0), "age": 0})
            next_id += 1

        result.append([(t["id"], t["box"]) for t in tracks])
    return result


def bridge_gaps(tracked, gap_max=40, dist_factor=1.5):
    n = len(tracked)
    per_id = {}
    for f, items in enumerate(tracked):
        for tid, box in items:
            per_id.setdefault(tid, {})[f] = box
    first = {tid: min(d) for tid, d in per_id.items()}
    last = {tid: max(d) for tid, d in per_id.items()}

    def center(b):
        return (b[0] + b[2] / 2, b[1] + b[3] / 2)

    added = [[] for _ in range(n)]
    for b_id in sorted(per_id, key=lambda t: first[t]):
        fb = first[b_id]
        boxb = per_id[b_id][fb]
        best, best_dist = None, None
        for a_id in per_id:                                  # früher beendete Spur suchen
            if a_id == b_id:
                continue
            gap = fb - last[a_id] - 1
            if gap < 1 or gap > gap_max:
                continue
            boxa = per_id[a_id][last[a_id]]
            (ax, ay), (bx, by) = center(boxa), center(boxb)
            dist = ((ax - bx) ** 2 + (ay - by) ** 2) ** 0.5
            if dist <= dist_factor * max(boxa[2], boxb[2]) and (best is None or dist < best_dist):
                best, best_dist = a_id, dist
        if best is not None:                                 # Lücke linear füllen
            la = last[best]
            boxa = per_id[best][la]
            for f in range(la + 1, fb):
                t = (f - la) / (fb - la)
                box = tuple(boxa[i] + t * (boxb[i] - boxa[i]) for i in range(4))
                added[f].append((best, box))
    return [tracked[f] + added[f] for f in range(n)]


def smooth_tracks(tracked, k=3, k_back=5):
    n = len(tracked)
    per_id = {}                                              # Kennung -> {Frame: Box}
    for f, items in enumerate(tracked):
        for tid, box in items:
            per_id.setdefault(tid, {})[f] = box

    out = [[] for _ in range(n)]
    for tid, d in per_id.items():
        first = min(d)
        for f in range(max(0, first - k_back), first):       # Anfang nach vorne verlängern
            d[f] = d[first]
        frames = sorted(d)
        arr = np.array([d[f] for f in frames], dtype=float)  # Spalten: x, y, w, h
        cx, cy = arr[:, 0] + arr[:, 2] / 2, arr[:, 1] + arr[:, 3] / 2
        for j, f in enumerate(frames):
            lo, hi = max(0, j - k), min(len(frames), j + k + 1)
            w, h = arr[lo:hi, 2].max(), arr[lo:hi, 3].max()  # größte Größe im Fenster
            mx, my = cx[lo:hi].mean(), cy[lo:hi].mean()      # gemittelte Mitte
            out[f].append((mx - w / 2, my - h / 2, w, h))
    return out


def dedupe(boxes_per_frame, overlap=0.6):
    out = []
    for boxes in boxes_per_frame:
        kept = []
        for b in sorted(boxes, key=lambda b: b[2] * b[3], reverse=True):
            if all(inter_over_smaller(b, kk) < overlap for kk in kept):
                kept.append(b)
        out.append(kept)
    return out

## 5. Hilfen zum Lesen des Ausschnitts (wie im 34G-Notebook)

`skip_to` liest die Frames davor nur und wirft sie weg, damit die Frame-Nummer sicher stimmt (`cap.set(POS_FRAMES)` ist bei MPEG-2 ungenau). `detect_clip` erkennt nur den Ausschnitt, speichert alle 25 Frames und setzt nach einem Abbruch fort.

In [ ]:
def skip_to(cap, index):
    """Springt zu Frame `index`, indem die Frames davor gelesen und verworfen werden."""
    for _ in range(index):
        if not cap.grab():
            raise RuntimeError("Video ist kürzer als erwartet")


def detect_clip(input_path, detector, start, end, out_file, save_every=25):
    """Erkennt die Frames start..end-1. Ergebnis: Liste mit einem Array [x, y, w, h, score] pro Frame."""
    faces = []
    if Path(out_file).exists():
        with open(out_file, "rb") as f:
            faces = pickle.load(f)
        print("Fortgesetzt ab Ausschnitt-Frame", len(faces))

    def save():
        Path(out_file).parent.mkdir(parents=True, exist_ok=True)
        with open(out_file, "wb") as f:
            pickle.dump(faces, f)

    cap = cv2.VideoCapture(str(input_path))
    t0 = time.time()
    try:
        skip_to(cap, start + len(faces))
        for _ in range(start + len(faces), end):
            ok, frame = cap.read()
            if not ok:
                break
            _, found = detector.detect(frame)
            faces.append(found)
            if len(faces) % save_every == 0:
                save()
                print(f"{len(faces)}/{end - start} Frames, {time.time() - t0:.0f} s")
    finally:
        save()
        cap.release()
    return faces

## 6. Kontrollen an zwei Frames

Wir laden zwei Frames in den Speicher (der erste und der mittlere des Ausschnitts) und machen damit zwei Prüfungen. Es werden nur Zahlen ausgegeben, keine Bilder.

**6a. Stimmt die Frame-Nummer?** 10G wird auf dem ersten Frame neu berechnet und mit der gespeicherten 10G-Erkennung für diese Nummer verglichen. Sind sie gleich, liegen alle Vergleiche auf denselben Bildern.

In [ ]:
class SCRFD:
    def __init__(self, model_path, input_size=(1280, 736), score_threshold=0.5, nms_threshold=0.4, pad_value=127):
        self.pad_value = pad_value
        self.in_w, self.in_h = input_size
        self.score_threshold = score_threshold
        self.nms_threshold = nms_threshold
        self.strides = [8, 16, 32]
        self.num_anchors = 2

        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]
        print("Ausgaben des Netzes:", len(self.output_names), "(6 = nur Boxen, 9 = mit Gesichtspunkten)")

    def detect(self, img_bgr):
        ih, iw = img_bgr.shape[:2]

        # 1. Letterbox
        scale = min(self.in_w / iw, self.in_h / ih)
        new_w, new_h = int(iw * scale), int(ih * scale)
        canvas = np.full((self.in_h, self.in_w, 3), self.pad_value, dtype=np.uint8)
        canvas[:new_h, :new_w] = cv2.resize(img_bgr, (new_w, new_h))

        # 2. Netz: (Pixel - 127.5) / 128, BGR -> RGB
        blob = cv2.dnn.blobFromImage(canvas, 1.0 / 128, (self.in_w, self.in_h),
                                     (127.5, 127.5, 127.5), swapRB=True)
        outs = self.session.run(self.output_names, {self.input_name: blob})

        boxes, scores = [], []
        for idx, stride in enumerate(self.strides):
            s = outs[idx].reshape(-1)                      # Scores
            d = outs[idx + 3].reshape(-1, 4) * stride      # Abstände in Pixeln

            # 3. Gitterpunkte dieses Maßstabs, je 2 Anker
            gh, gw = self.in_h // stride, self.in_w // stride
            ys, xs = np.mgrid[:gh, :gw]
            centers = np.stack([xs, ys], axis=-1).astype(np.float32).reshape(-1, 2) * stride
            centers = np.repeat(centers, self.num_anchors, axis=0)

            for k in np.where(s >= self.score_threshold)[0]:
                cx, cy = centers[k]
                x1, y1 = cx - d[k, 0], cy - d[k, 1]
                x2, y2 = cx + d[k, 2], cy + d[k, 3]
                boxes.append([x1 / scale, y1 / scale, (x2 - x1) / scale, (y2 - y1) / scale])
                scores.append(float(s[k]))

        if not boxes:
            return 0, np.empty((0, 5), dtype=np.float32)

        # 4. NMS
        idxs = np.array(cv2.dnn.NMSBoxes(boxes, scores, self.score_threshold, self.nms_threshold)).reshape(-1)
        faces = np.array([[*boxes[i], scores[i]] for i in idxs], dtype=np.float32)
        return len(faces), faces

In [ ]:
with open(FACES_10G_FILE, "rb") as f:
    faces_10g_all = pickle.load(f)
print("10G-Erkennungen für", len(faces_10g_all), "Frames geladen")

mid = (CLIP_START + CLIP_END) // 2
cap = cv2.VideoCapture(str(INPUT_VIDEO))
skip_to(cap, CLIP_START)
ok1, frame_first = cap.read()
for _ in range(mid - CLIP_START - 1):
    cap.grab()
ok2, frame_mid = cap.read()
cap.release()
assert ok1 and ok2

detector_10g = SCRFD(MODEL_10G, input_size=(1280, 736), score_threshold=0.2, pad_value=0)
_, neu = detector_10g.detect(frame_first)
alt = faces_10g_all[CLIP_START]
if len(alt) == 0:
    gleich = len(neu) == 0
else:
    gleich = len(neu) == len(alt) and np.allclose(neu[np.argsort(-neu[:, 4])], alt[np.argsort(-alt[:, 4])], atol=1e-2)
print(f"Frame {CLIP_START}: neu {len(neu)} Funde, gespeichert {len(alt)} Funde -> gleich: {gleich}")
assert gleich, "Frame-Nummer passt nicht zur gespeicherten Erkennung - Ausschnitt nicht vergleichbar"

**6b. RGB oder BGR?** Dasselbe Bild, zweimal durch den Kopfdetektor, einmal mit jeder Farbreihenfolge. Der höhere Bestwert zeigt die richtige Reihenfolge. Das Beispielskript des Autors sagt BGR, wir erwarten RGB.

In [ ]:
for label, rgb in [("RGB", True), ("BGR", False)]:
    det = HeadDetector(MODEL_HEAD, score_threshold=0.2, rgb=rgb)
    n, found = det.detect(frame_mid)
    best = float(found[:, 4].max()) if n else 0.0
    print(f"Frame {mid}, {label}: {n} Kopfboxen (Score >= 0,2), bester Score {best:.2f}")

del frame_first, frame_mid     # Bilder nicht im Speicher behalten

## 7. Kopfdetektor auf dem Ausschnitt laufen lassen
Score-Schwelle 0,2 (alles ab 0,2 wird gespeichert, die Auswahl kommt im Tracking). Der Lauf ist viel schneller als SCRFD-34G, weil das Netz nur 640 × 640 verarbeitet. Den größten Teil der Zeit braucht das Vorspulen zu Frame `CLIP_START`.

In [ ]:
detector_head = HeadDetector(MODEL_HEAD, score_threshold=0.2)

t0 = time.time()
heads = detect_clip(INPUT_VIDEO, detector_head, CLIP_START, CLIP_END, HEADS_FILE)
faces_10g = faces_10g_all[CLIP_START:CLIP_END]
print(f"Frames im Ausschnitt: Kopf {len(heads)} | 10G {len(faces_10g)} | {time.time() - t0:.0f} s")
assert len(heads) == len(faces_10g)

## 8. Plausibilität: Liegen die sicheren 10G-Gesichter in einer Kopfbox?

Wenn der Kopfdetektor Köpfe findet, muss die **Mitte eines sicheren Gesichts** (10G-Score ≥ 0,5) in einer Kopfbox liegen. Ist das fast immer so, passen die beiden Modelle zusammen und die Box-Umrechnung stimmt. Ist es oft nicht so, stimmt etwas nicht (Koordinaten, Farben) oder der Kopfdetektor findet einen anderen Teil.

In [ ]:
def face_in_head(faces_list, heads_list, face_thr=0.5, head_thr=0.3):
    hit = total = 0
    for fa, he in zip(faces_list, heads_list):
        for f in fa:
            if f[4] < face_thr:
                continue
            total += 1
            cx, cy = f[0] + f[2] / 2, f[1] + f[3] / 2
            if any(h[4] >= head_thr and h[0] <= cx <= h[0] + h[2] and h[1] <= cy <= h[1] + h[3] for h in he):
                hit += 1
    return hit, total


for head_thr in (0.5, 0.3, 0.2):
    hit, total = face_in_head(faces_10g, heads, head_thr=head_thr)
    print(f"Kopfbox ab Score {head_thr}: {hit} von {total} sicheren 10G-Gesichtern liegen mit ihrer Mitte in einer Kopfbox")

## 9. Vergleich der rohen Erkennungen

Pro Frame und Ansicht zählt der beste Score (0 = nichts gefunden). Die Scores der zwei Modelle sind nicht direkt vergleichbar. Wichtig ist, **ob** in einem Frame etwas gefunden wird, nicht die genaue Zahl.

In [ ]:
def in_half(box, half):
    cx = box[0] + box[2] / 2
    return cx < SPLIT if half == "links" else cx >= SPLIT


def best_per_frame(faces, half):
    return np.array([max((float(b[4]) for b in fr if in_half(b, half)), default=0.0) for fr in faces])


n = CLIP_END - CLIP_START
print(f"Ausschnitt: Frame {CLIP_START}-{CLIP_END - 1} (Sekunde {CLIP_START / FPS:.1f}-{CLIP_END / FPS:.1f}), {n} Frames\n")
for half in ("links", "rechts"):
    for name, fa in (("Gesicht 10G", faces_10g), ("Kopf YOLO11", heads)):
        best = best_per_frame(fa, half)
        print(f"{half:6s} {name:12s}: Frames mit Fund >=0.5: {(best >= 0.5).sum():3d} | >=0.3: {(best >= 0.3).sum():3d} | "
              f">=0.2: {(best >= 0.2).sum():3d} | Median des Bestwerts: {np.median(best):.2f}")
    print()

print("Verlauf (bester Score je Ansicht, Maximum über je 10 Frames):")
print("Frame  | links Gesicht10G  Kopf | rechts Gesicht10G  Kopf")
for st in range(0, n, 10):
    row = []
    for half in ("links", "rechts"):
        for fa in (faces_10g, heads):
            row.append(best_per_frame(fa[st:st + 10], half).max())
    print(f"{CLIP_START + st:6d} |             {row[0]:.2f}  {row[1]:.2f} |              {row[2]:.2f}  {row[3]:.2f}")

## 10. Nachbearbeitung und Zählung der Frames ohne Box

Dieselbe Kette wie bei 10G, auf die Kopfboxen angewendet, mit drei Startschwellen (0,5 / 0,4 / 0,3), weil die Scores anders skaliert sind.

**Rand (`padding`):** Bei Gesichtsboxen verwenden wir 0,5, damit Stirn, Haare und Kinn mit abgedeckt sind. Eine Kopfbox enthält das schon. Für Köpfe nehmen wir deshalb nur **0,15** als Reserve gegen Zittern. Das ist eine Annahme von mir, kein gemessener Wert. Du kannst `PADDING_HEAD` ändern.

- **Frames ohne Box:** dort ist nichts verpixelt (Ziel: 0).
- **Fläche:** Median und Maximum der verpixelten Fläche in Prozent des Bildes.

In [ ]:
PADDING_FACE = 0.5
PADDING_HEAD = 0.15


def pipeline(faces, start_threshold=0.5):
    t = track_with_ids(faces, start_threshold=start_threshold, low_threshold=0.2, max_age=8, near_threshold=0.2)
    t = bridge_gaps(t, gap_max=40)
    return dedupe(smooth_tracks(t, k=3, k_back=5))


def area_percent(boxes_per_frame, padding, shape=(1080, 1920)):
    h, w = shape
    out = []
    for boxes in boxes_per_frame:
        mask = np.zeros((h, w), dtype=bool)
        for x, y, bw, bh in boxes:
            mask[int(max(0, y - padding * bh)):int(min(h, y + bh + padding * bh)),
                 int(max(0, x - padding * bw)):int(min(w, x + bw + padding * bw))] = True
        out.append(100.0 * mask.mean())
    return np.array(out)


variants = {
    "Gesicht 10G, Start 0,5 (bisher)": (faces_10g, 0.5, PADDING_FACE),
    "Kopf YOLO11, Start 0,5": (heads, 0.5, PADDING_HEAD),
    "Kopf YOLO11, Start 0,4": (heads, 0.4, PADDING_HEAD),
    "Kopf YOLO11, Start 0,3": (heads, 0.3, PADDING_HEAD),
}
results = {}
print(f"{'Variante':34s} | Rand | ohne Box: gesamt  links  rechts | Fläche Median / Max")
for name, (fa, thr, pad) in variants.items():
    boxes = pipeline(fa, start_threshold=thr)
    results[name] = boxes
    none_all = sum(1 for b in boxes if not b)
    none_l = sum(1 for b in boxes if not any(in_half(x, "links") for x in b))
    none_r = sum(1 for b in boxes if not any(in_half(x, "rechts") for x in b))
    a = area_percent(boxes, pad)
    print(f"{name:34s} | {pad:.2f} |           {none_all:5d} {none_l:6d} {none_r:7d} | {np.median(a):.1f} % / {a.max():.1f} %")

## 11. Optional: Ausschnitt als Video

Gerendert werden „Gesicht 10G, Start 0,5“ (der bisherige Stand) und eine Kopf-Variante (`HEAD_KEY`, standardmäßig Start 0,5; nach der Tabelle oben anpassen). Sieh dir an:
- Sitzt die Kopfbox auf dem Kopf des Patienten? Auch beim Anfall, auch in der schwarz-weißen Ansicht?
- Werden Personen verpixelt, die nicht der Patient sind (Pfleger)? Das ist gewollt oder nicht, je nach Datenschutzkonzept.
- Gibt es Boxen auf Dingen, die keine Köpfe sind (Kissen, Hände, Geräte)?

In [ ]:
def render_clip(input_path, output_path, boxes_per_frame, start, block_px=22, padding=0.5):
    """Wie render_mask aus dem 10G-Notebook, aber nur für den Ausschnitt ab Frame `start`."""
    cap = cv2.VideoCapture(str(input_path))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    bw, bh = max(1, width // block_px), max(1, height // block_px)
    written = 0
    try:
        skip_to(cap, start)
        for boxes in boxes_per_frame:
            ok, frame = cap.read()
            if not ok:
                break
            mask = np.zeros((height, width), dtype=bool)
            for x, y, w, h in boxes:
                mask[int(max(0, y - padding * h)):int(min(height, y + h + padding * h)),
                     int(max(0, x - padding * w)):int(min(width, x + w + padding * w))] = True
            if mask.any():
                small = cv2.resize(frame, (bw, bh), interpolation=cv2.INTER_AREA)
                pix = cv2.resize(small, (width, height), interpolation=cv2.INTER_NEAREST)
                frame[mask] = pix[mask]
            writer.write(frame)
            written += 1
    finally:
        cap.release()
        writer.release()
    return written


RENDER_CLIPS = True
HEAD_KEY = "Kopf YOLO11, Start 0,5"
if RENDER_CLIPS:
    for tag, key, pad in [("gesicht10g", "Gesicht 10G, Start 0,5 (bisher)", PADDING_FACE), ("kopf", HEAD_KEY, PADDING_HEAD)]:
        out = OUT_DIR / f"VID_76_clip_{CLIP_START}_{CLIP_END}_{tag}.mp4"
        print(tag, render_clip(INPUT_VIDEO, out, results[key], CLIP_START, padding=pad), "Frames ->", out.name)

## 12. Auswertung (nach dem Lauf ausfüllen)

**Wie du die Zahlen einordnest:**
- **Frames ohne Box sinken stark** (Kopf gegen Gesicht 10G) und im Video sitzen die Boxen auf dem Kopf des Patienten: Der Kopfdetektor löst das Problem für diesen Ausschnitt.
- **Frames ohne Box sinken, aber die Fläche steigt stark oder es gibt Boxen auf Nicht-Köpfen:** Die Schwelle oder der Rand muss angepasst werden, oder das Modell ist zu unspezifisch.
- **Auch der Kopfdetektor findet in der schwarz-weißen Ansicht nichts:** CrowdHuman enthält kaum Nachtaufnahmen. Dann eher ein anderes Modell oder Feintuning auf eigenen Frames.
- Nur **ein Ausschnitt** von 6 Sekunden: ein Hinweis, kein Beweis. Danach mindestens zwei weitere Lücken testen (Frames 7789–7902, 6742–6801, 5576–5621).

**Ergebnis:** _(hier eintragen)_